In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import panel as pn

from notebooks_dani.eicu_db_connection import DatabaseConnection
from dani_plotting import daniplot
import palettable

import holoviews as hv
import hvplot.pandas  # noqa
from holoviews import opts

from loguru import logger

hv.extension("bokeh")

%matplotlib inline

In [ ]:
notebook_name = "apache_all"

In [ ]:
connection = DatabaseConnection()
engine = connection.engine
query_schema = connection.query_schema

In [ ]:
INTER_DATA_PATH = "../../intermediary_data/"
print(f"Loading and saving data from/to {Path(INTER_DATA_PATH).resolve()}")


# APS Vars

In [ ]:
query_result = query_schema + """
select *
from apacheapsvar
"""

aps_var_csv_path = Path(INTER_DATA_PATH) / Path("apache_aps_var.csv")

if aps_var_csv_path.is_file():
    df_apache_aps = pd.read_csv(aps_var_csv_path)
    logger.info(f"Loading data from local csv.")
else:
    df_apache_aps = pd.read_sql_query(query_result, engine)
    df_apache_aps.to_csv(path_or_buf=aps_var_csv_path)
    logger.info(f"Loading data from database.")

df_apache_aps_sample = df_apache_aps.head(n=1)

# Saving another df w/o ids
df_apache_aps_with_ids = df_apache_aps.copy()
df_apache_aps = df_apache_aps_with_ids.drop(columns=["apacheapsvarid", "patientunitstayid"])

### Taking a look at how often APS variables have been determined for each patientunitstayid



In [ ]:
df_apache_aps_patient_grouped = df_apache_aps_with_ids.groupby("patientunitstayid")

In [ ]:
df_patient_apache_aps_measurements = df_apache_aps_patient_grouped.patientunitstayid.value_counts()
df_patient_apache_aps_measurements.describe()

**This raises the question if the apache aps measurements are really conducted once during an ICU stay?**


## Taking a look at the statistics of each variable (occurances, mean, median, variability)


In [ ]:
aps_vars = df_apache_aps.columns.values.tolist()

df_apache_aps.describe()

I should plot descriptive statistics in the next steps, to get a better glimpse at the outliers, somebody managed to urinate 269 L/24h considering the outlier.


In [ ]:
fig_box, axs_box, font = daniplot.plot_config_single(num_row=1, num_cols=1, width_cm=50, height_cm=20)

df_apache_aps.plot(subplots=True, kind="box", ax=axs_box, sharey=False, rot=59)  #, sharex=False)

### The Apache APS Vars can be categorized in following fashon


In [ ]:
import itertools

vars_intervention = ["intubated", "vent", "dialysis"]
vars_neuro = ["eyes", "motor", "verbal"]
vars_comorb = ["meds"]
vars_physio = ["urine", "temperature", "respiratoryrate", "heartrate", "meanbp"]
vars_blood_serum = ["wbc", "sodium", "hematocrit", "creatinine", "albumin", "glucose", "bilirubin"]
vars_blood_gas = ["ph", "pao2", "pco2", "fio2", "bun"]

vars_all = [vars_intervention, vars_neuro, vars_comorb, vars_physio, vars_blood_serum, vars_blood_gas]
print(f"Number of sorted APS variables={len(list(itertools.chain.from_iterable(vars_all)))}")

df_aps_neuro = df_apache_aps[vars_neuro]
df_aps_comorb = df_apache_aps[vars_comorb]
df_aps_physio = df_apache_aps[vars_physio]
df_aps_blood_hema = df_apache_aps[vars_blood_serum]
df_aps_blood_gas = df_apache_aps[vars_blood_gas]

df_aps_list = [df_aps_neuro, df_aps_comorb, df_aps_physio, df_aps_blood_hema, df_aps_blood_gas]

### Distribution of Physiological APS Vars


In [ ]:

df_physio_wo_urine = df_aps_physio.drop(columns=["urine"], axis=1)
box_plot_physio = df_physio_wo_urine.hvplot(
    kind='box',
    title='Physiological Vars',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
    # responsive=True,
    #min_height=400
)

violin_plot_physio = df_physio_wo_urine.hvplot(
    kind='violin',
    title='Physiological Vars',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

violin_plot_urine = df_aps_physio["urine"].hvplot(
    kind='violin',
    title='Physiological Vars (Urine)',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

In [ ]:
#pane_box_physio = pn.Card(pn.panel(box_plot_physio))
pane_box_physio = pn.panel(box_plot_physio)
#pane_violin_physio = pn.Card(pn.panel(violin_plot_physio))
pane_violin_physio = pn.panel(violin_plot_physio)
#pane_violin_urine = pn.Card(pn.panel(violin_plot_urine))
pane_violin_urine = pn.panel(violin_plot_urine)

pn.Row(pane_box_physio, pane_violin_physio, pane_violin_urine, sizing_mode='stretch_both', scroll=True)

### Distribution of Blood Hematology Vars

In [ ]:
#df_physio_wo_urine = df_aps_physio.drop(columns=["urine"], axis=1)
box_plot_hema = df_aps_blood_hema.drop(columns=["glucose"], axis=1).hvplot(
    kind='box',
    title='Hematological Vars',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

violin_plot_hema = df_aps_blood_hema.drop(columns=["glucose"], axis=1).hvplot(
    kind='violin',
    title='Hematological Vars',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

violin_plot_glucose = df_aps_blood_hema["glucose"].hvplot(
    kind='violin',
    title='Hematological Vars',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

pane_box_hema = pn.Card(pn.panel(box_plot_hema))
pane_violin_hema = pn.Card(pn.panel(violin_plot_hema))
pane_violin_glucose = pn.Card(pn.panel(violin_plot_glucose))

pn.Row(pane_box_hema, pane_violin_hema, pane_violin_glucose, sizing_mode='stretch_both', scroll=True)

### Distribution of Blood Gas Vars

In [ ]:
#box_plot_hema = df_aps_blood_hema.drop(columns=["glucose"], axis=1).hvplot(
box_plot_gas = df_aps_blood_gas.drop(columns=["pao2"], axis=1).hvplot(
    kind='box',
    title="Blood Gas",
    x='index',
    legend='bottom_right',
    widget_location='bottom',
)

violin_plot_gas = df_aps_blood_gas.drop(columns=["pao2"], axis=1).hvplot(
    kind='violin',
    title="Blood Gas",
    x='index',
    legend='bottom_right',
    widget_location='bottom',
)

violin_plot_pao2 = df_aps_blood_gas["pao2"].hvplot(
    kind='violin',
    title="Blood Gas",
    x='index',
    legend='bottom_right',
    widget_location='bottom',
)

pane_box_gas = pn.Card(pn.panel(box_plot_gas))
pane_violin_gas = pn.Card(pn.panel(violin_plot_gas))
pane_violin_pao2 = pn.Card(pn.panel(violin_plot_pao2))

pn.Row(pane_box_gas, pane_violin_gas, pane_violin_pao2, sizing_mode='stretch_both', scroll=True)

The descriptive statistics from above include icu stays that may not feature specific measurements, like temperature=-1.
**I think we should remove those with regard to the ranges at least for the statistics of the parameters so they don't skew the data as much.
How should we deal with missing values with regard to the model? Exclude them? Educated guess? Include invalid values?**


# Apache Patient Results

In [ ]:
query_result = query_schema + """
select *
from apachepatientresult
"""

ap_pat_results_filename = "apache_patient_result.csv"
ap_pat_results_csv_path = Path(INTER_DATA_PATH) / Path(ap_pat_results_filename)

if ap_pat_results_csv_path.is_file():
    logger.info(f"Loading data from local csv.")
    df_results_all = pd.read_csv(ap_pat_results_csv_path)

else:
    logger.info(f"Loading data from database.")
    df_results_all = pd.read_sql_query(query_result, engine)
    df_results_all.to_csv(path_or_buf=ap_pat_results_csv_path)


#df_results_all = pd.read_sql_query(query_result, engine)

In [ ]:
df_results_all.head(n=10)

There are multiple predictions for individual patientunitstayid's. Is this due to different Apache versions? I.e. IV and IVa? 
 

In [ ]:
number_apacheids_results_total = df_results_all["apachepatientresultsid"].unique().size
print(f"The total number of unique apache ids is \n {number_apacheids_results_total}")

In [ ]:
number_patientstays_results_total = df_results_all["patientunitstayid"].unique().size
print(
    f"The total number of unique patient stays is \n {number_patientstays_results_total}.\n However, this doesn't mean that we can use all entries. We need the entries that container the Apache Scores, prediction, and actual mortality.")

In [ ]:
ratio_apacheids_patientstays = number_apacheids_results_total / number_patientstays_results_total
ratio_apacheids_patientstays

Okay it's relatively safe to assume that the eICU System calculates two different types of Apache IV.
**It's probably the best idea to do all performance evaluation with regard the more up to date Score**


In [ ]:
df_results_apache_IVa_all = df_results_all[df_results_all["apacheversion"] == "IVa"]
df_results_apache_IVa_all.head()

In [ ]:
df_results_valid_pred_mortality = df_results_apache_IVa_all[df_results_apache_IVa_all["predictedicumortality"] != -1]
print(f"The total number of valid apache predictions is \n {df_results_valid_pred_mortality.shape[0]}")

In [ ]:
ap_pat_results_valid_mortality_filename = "apache_patient_result_valid_mortality.csv"
ap_pat_results_valid_mortality_csv_path = Path(INTER_DATA_PATH) / Path(ap_pat_results_valid_mortality_filename)

if ap_pat_results_valid_mortality_csv_path.is_file():
    logger.info(f"Data is already present in csv, skipping save.")

else:
    logger.info(f"No csv file found, saving ")
    #df_results_all = pd.read_sql_query(query_result, engine)
    df_results_valid_pred_mortality.to_csv(path_or_buf=ap_pat_results_valid_mortality_csv_path)


Filtering the dataframe in the debugger reveals, that are not -1 values for the column "actualicumortality", which indicates that for every entry with a valid prediction there's also a valid "actualicumortality" state

In [ ]:
scatter_pred_mort = df_results_valid_pred_mortality[["predictedicumortality", "actualicumortality"]].hvplot(
    by=['actualicumortality'],
    kind='scatter',
    x='index',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

violin_pred_mort = df_results_valid_pred_mortality[["predictedicumortality", "actualicumortality"]].hvplot(
    by=['actualicumortality'],
    kind='violin',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)

box_pred_mort = df_results_valid_pred_mortality[["predictedicumortality", "actualicumortality"]].hvplot(
    by=['actualicumortality'],
    kind='box',
    legend='bottom_right',
    widget_location='bottom',
    shared_axes=False
)



In [ ]:
def plot_panel_row_in_notebook(plot_list: list) -> pn.Row:
    pane_list = [pn.Card(pn.panel(plot)) for plot in plot_list]
    row = pn.Row(*pane_list, sizing_mode='stretch_both', scroll=True)

    return row


In [ ]:
row_pred_mort = plot_panel_row_in_notebook([scatter_pred_mort, box_pred_mort, violin_pred_mort])
row_pred_mort


I'd like to see the histogram of the predicted mortality for folks that expired, since this show really obvious why apache alone is "bad".


**Number of patient that made it out alive:**

In [ ]:
number_alive = df_results_valid_pred_mortality[df_results_valid_pred_mortality["actualicumortality"] == "ALIVE"].shape[
    0]
number_alive

**Number of patients that expired:**


In [ ]:
number_expired = \
df_results_valid_pred_mortality[df_results_valid_pred_mortality["actualicumortality"] == "EXPIRED"].shape[0]
number_expired

In [ ]:

mortality_stacked_hist = df_results_valid_pred_mortality[["predictedicumortality", "actualicumortality"]].hvplot(
    by='actualicumortality',
    kind='hist',
    y='predictedicumortality',
    legend='bottom_right',
    widget_location='bottom',
)

mortality_grouped_hist = df_results_valid_pred_mortality[["predictedicumortality", "actualicumortality"]].hvplot(
    by=['actualicumortality'],
    groupby=['actualicumortality'],
    kind='hist',
    y=['predictedicumortality'],
    legend='bottom_right',
    widget_location='bottom',
)
mortality_stacked_hist


In [ ]:
mortality_grouped_hist

In [ ]:
df_mort_loss = df_results_valid_pred_mortality[
    ["predictedicumortality", "actualicumortality", "predictediculos", "actualiculos"]]

mort_los_scatter = df_mort_loss.hvplot(
    by='actualicumortality',
    groupby='actualicumortality',
    kind='scatter',
    x='predictediculos',
    y='actualiculos',
    legend='bottom_right',
    widget_location='bottom',
)

row_mort_los_scatter = plot_panel_row_in_notebook([mort_los_scatter])
row_mort_los_scatter

It seems that the range of LOS for alive patients is much more spread out, which makes sense, since dying early is a cap on LOS.

In [ ]:
bars_alive_expired = hv.Bars(df_mort_loss, kdims=["actualicumortality"]).aggregate(function=np.sum)
bars_alive_expired = bars_alive_expired.opts(shared_axes=False)
bars_alive_expired

In [ ]:
df_mort_loss.head(n=1)

In [ ]:
#hv.help(hv.BoxWhisker)
box_pred_los_mort = hv.BoxWhisker(data=df_mort_loss, kdims=["actualicumortality"], vdims=["predictediculos"])
box_actual_los_mort = hv.BoxWhisker(data=df_mort_loss, kdims="actualicumortality", vdims="actualiculos")

los_mort_boxes = (box_pred_los_mort + box_actual_los_mort).opts(shared_axes=False)
los_mort_boxes

# APS Vars vs. Expired/Alive

In [ ]:
df_apache_aps_with_ids.head()
df_apache_aps_with_ids.shape

In [ ]:
df_results_valid_pred_mortality.head()
df_results_valid_pred_mortality.shape

### Combining the APS vars with the valid Apache results table (only results where the predicted mortality has been determined) 

In [ ]:

df_aps_vars_ap_results = pd.merge(df_apache_aps_with_ids, df_results_valid_pred_mortality, on="patientunitstayid",
                                  how="inner")
df_aps_vars_ap_results.head()
df_aps_vars_ap_results.drop(columns=["Unnamed: 0_x", "Unnamed: 0_y"], axis=1, inplace=True)


In [ ]:
aps_var_results_valid_mortality_filename = "apache_aps_var_results_valid_mortality.csv"
aps_var_results_valid_mortality_csv_path = Path(INTER_DATA_PATH) / Path(aps_var_results_valid_mortality_filename)

if aps_var_results_valid_mortality_csv_path.is_file():
    logger.info(f"Data is already present in csv, skipping save.")

else:
    logger.info(f"No csv file found, saving ")
    #df_results_all = pd.read_sql_query(query_result, engine)
    df_aps_vars_ap_results.to_csv(path_or_buf=aps_var_results_valid_mortality_csv_path)

In [ ]:
df_aps_vars_ap_results_correlation = df_aps_vars_ap_results.corr(method="pearson", numeric_only=True)

aps_var_results_valid_mortality_correlation_filename = "apache_aps_var_results_correlation.csv"
aps_var_results_valid_mortality_correlation_csv_path = Path(INTER_DATA_PATH) / Path(
    aps_var_results_valid_mortality_correlation_filename)

if aps_var_results_valid_mortality_correlation_csv_path.is_file():
    logger.info(f"Data is already present in csv, skipping save.")

else:
    logger.info(f"No csv file found, saving ")
    #df_results_all = pd.read_sql_query(query_result, engine)
    df_aps_vars_ap_results_correlation.to_csv(path_or_buf=aps_var_results_valid_mortality_correlation_csv_path)

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(df_aps_vars_ap_results_correlation,
            #cmap=sns.diverging_palette(220, 10, as_cmap=True),
            cmap=sns.color_palette("icefire", as_cmap=True),
            vmin=-1.0, vmax=1.0,
            square=True, ax=ax)

filename_corr = "APS Vars Apache Results Correlation"
plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

In [ ]:
df_aps_vars_ap_results_correlation_threshold_40 = df_aps_vars_ap_results_correlation.applymap(
    lambda x: 0 if (abs(x)) < 0.4 else x)
df_aps_vars_ap_results_correlation_threshold_40

In [ ]:
fig_corr_2, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(df_aps_vars_ap_results_correlation_threshold_40,
            #cmap=sns.diverging_palette(220, 10, as_cmap=True),
            cmap=sns.color_palette("icefire", as_cmap=True),
            vmin=-1.0, vmax=1.0,
            square=True, ax=ax)

filename_corr = "APS Vars Apache Results Correlation Threshold 0.4"
plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

In [ ]:
#data = [(i, j, df_aps_vars_ap_results_correlation.iloc[i, j]) for i in range(df_aps_vars_ap_results_correlation.shape[0]) for j in range(df_aps_vars_ap_results_correlation.shape[1])]

data = [(df_aps_vars_ap_results_correlation.index[i], df_aps_vars_ap_results_correlation.columns[j],
         df_aps_vars_ap_results_correlation.iloc[i, j])
        for i in range(df_aps_vars_ap_results_correlation.shape[0])
        for j in range(df_aps_vars_ap_results_correlation.shape[1])]

heatmap_data = pd.DataFrame(data, columns=['Variable1', 'Variable2', 'Correlation'])

# Create a Heatmap
heatmap = hv.HeatMap(heatmap_data, kdims=['Variable1', 'Variable2'], vdims='Correlation').opts(tools=['hover'],
                                                                                               colorbar=True, width=700,
                                                                                               height=700,
                                                                                               cmap='icefire',
                                                                                               xaxis='bottom',
                                                                                               xrotation=65, xlabel='',
                                                                                               ylabel='')
heatmap

In [ ]:
def filtered_heatmap(threshold):
    filtered_data = heatmap_data[heatmap_data['Correlation'].abs() >= threshold]
    heatmap_filt = hv.HeatMap(filtered_data, kdims=['Variable1', 'Variable2'], vdims='Correlation').opts(
        tools=['hover'], colorbar=True, width=700, height=700, cmap='icefire', xaxis='bottom', xrotation=65, xlabel='',
        ylabel='')
    return heatmap_filt


In [ ]:
threshold_slider = pn.widgets.FloatSlider(name='Correlation Threshold Heatmap', start=0, end=1, step=0.01, value=0.5)
interactive_heatmap = pn.interact(filtered_heatmap, threshold=threshold_slider)
#interactive_heatmap


In [ ]:
import panel as pn
from panel.template import DarkTheme

#pn.extension(sizing_mode="stretch_both")
pn.extension(
    "tabulator", design="material", template="material", loading_indicator=True
)

ACCENT = "#BB2649"
#panel_page = pn.template.BootstrapTemplate(title="Apache Data Exploration", theme=DarkTheme, header_color="red")
#panel_page = pn.template.FastListTemplate(
panel_page = pn.template.BootstrapTemplate(
    sidebar=[threshold_slider],
    #sidebar_width=33,
    title="Apache Data Exploration",
    #accent_base_color=ACCENT,
    header_background=ACCENT,
    #prevent_collision=True,
    #save_layout=True,
    #theme_toggle=True,
    #theme="dark"
    #theme=DarkTheme,
    #row_height=160,
    #meta_author="Daniel Breucker",
)


panel_page.main.append(pn.pane.Markdown("""# Apache Acute Physiology Score (APS) Variables"""))
panel_page.main.append(pn.Row(box_plot_physio, violin_plot_physio, pane_violin_urine, sizing_mode="stretch_width"))
panel_page.main.append(pn.Row(box_plot_hema, violin_plot_hema, violin_plot_glucose, sizing_mode="stretch_width"))
panel_page.main.append(pn.Row(box_plot_gas, violin_plot_gas, violin_plot_pao2, sizing_mode="stretch_width"))
panel_page.main.append(pn.pane.Markdown("""# Apache Results"""))
panel_page.main.append(pn.pane.Markdown("""## Apache Predicted Mortality vs. Actual Mortality"""))
panel_page.main.append(pn.Row(scatter_pred_mort, box_pred_mort, violin_pred_mort, sizing_mode="stretch_width"))
panel_page.main.append(pn.pane.Markdown("""## Apache Length of Stay (LOS) Expired vs. Alive"""))
panel_page.main.append(
    pn.Row(mort_los_scatter, bars_alive_expired, mortality_grouped_hist, sizing_mode="stretch_width"))
panel_page.main.append(pn.Row(los_mort_boxes, sizing_mode="stretch_width"))
panel_page.main.append(pn.pane.Markdown("""# Apache Predictions, Mortality and APS Pearson Correlation"""))
panel_page.main.append(
    pn.Row(pn.bind(filtered_heatmap, threshold=threshold_slider),  sizing_mode="stretch_width"))

panel_page.show()

In [ ]:
from bokeh.resources import INLINE

panel_page.save(f'panel_exports/apache_aps_vars_results.html', resources=INLINE)